# TlBr 単素子 波形解析ノートブック（picodaq 版）

`tlbr_waveform_analysis.ipynb` の流れはそのままに、処理を picodaq ライブラリに置き換えたものです。
GUI の QL、`picodaq-analyze`、このノートブックが同じ関数を使うので、どこで計算しても結果は一致します。

1. 設定（データの場所と各段のパラメータ）
2. 生波形を見る（極性・立ち上がり・減衰）
3. 減衰時定数 τ（pole-zero）
4. 台形整形と波高の取り出し窓
5. 整形パラメータのスキャン（光電ピークの分解能で決める）
6. 全イベントの処理 → イベントファイル（FITS）と品質カット
7. スペクトル（anode / cathode）
8. CAR による深さ補正
9. エネルギー校正（Ba-133 / Eu-152 / Cs-137）
10. 校正済みスペクトルと分解能
11. 保存

同じ処理は CLI でも一括で実行できます：`picodaq-analyze run analysis.yaml`

## 1. 設定

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import picodaq
from picodaq import (EnergyCalibration, DepthCorrection, EventFile, LINES, TrapezoidShaper,
                     WaveformReader, calibrate, fit_peak, fit_tail_tau, histogram, load_config,
                     processors_from_config, rise_time, tune_tau)
from picodaq.config import deep_merge
from picodaq.events import _expand
from picodaq.plots import calibration_figure, depth_figure, spectra_figure

%matplotlib inline
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True, "grid.alpha": 0.3, "font.size": 10})
print("picodaq", picodaq.__version__)

In [ ]:
# ---- データ: 線源名 -> HDF5 ファイルまたはディレクトリ ------------------------------
# データはリポジトリの隣の data/ を見る（このノートブックは picodaq/notebooks/ にある）。
# 別の場所なら環境変数 PICODAQ_DATA で指定する。
import os
DATA_ROOT = Path(os.environ.get("PICODAQ_DATA", "../../data")).expanduser()
DATA = {
    "Ba133": [DATA_ROOT / "20260909/run_Ba/data"],
    # "Eu152": [DATA_ROOT / "20260909/run_Eu/data"],
    # "Cs137": [DATA_ROOT / "20260909/run_Cs137/data"],
}
OUT = Path("../analysis/20260909")   # picodaq/analysis/ (git には入らない)

# ---- 波形処理: ファイルに保存された設定にこれを上書きする (None なら上書きなし) -------
# 旧 run_acqusition.py のファイルには processing が無いので、少なくとも tau は指定する
PROCESSING = {
    "shaper": {"rise_us": 15.0, "flat_us": 16.0},
    "flattop_window": [0.6, 0.9],   # フラットトップの後半だけを平均（§4）
    "channels": {"A": {"polarity": 1, "tau_us": 55.0}, "B": {"polarity": -1, "tau_us": 17.0}},
}
CUTS = {"bl_rms_max": 0.010, "drift_max": 0.050, "late_max": 0.050}
DEPTH = dict(source="Ba133", amp_window=(0.06, 0.14), car_range=(0.25, 1.40), car_bins=12,
             car_ref=1.0, degree=3)
CAL = dict(model="linear", approx=None, min_amp=0.015)
N_PREVIEW = 500
MAX_EVENTS = None          # 1ファイルあたりの上限（試すときだけ）

OUT.mkdir(parents=True, exist_ok=True)
files = {src: _expand(p) for src, p in DATA.items()}
for src, fs in files.items():
    print(src, [f.name for f in fs])

## 2. 生波形を見る

In [ ]:
SRC0 = DEPTH["source"]
reader = WaveformReader(files[SRC0][0])
print(reader)
proc_file = load_config(reader.config)["processing"]
processing = deep_merge(proc_file, PROCESSING or {})
proc = processors_from_config(processing, reader.channels, reader.dt, reader.trigger_index)
ANODE, CATHODE = processing["anode"], processing["cathode"]
t_us = reader.time * 1e6
dt, ti = reader.dt, reader.trigger_index

raw = {ch: proc[ch].correct(reader.waveforms(ch, slice(0, N_PREVIEW)))[0] for ch in reader.channels}
order = np.argsort(raw[ANODE][:, ti:ti + int(30e-6 / dt)].mean(1))[::-1]
fig, axes = plt.subplots(2, 3, figsize=(13, 6), sharex=True)
for ax, i in zip(axes.ravel(), order[:6]):
    for ch in reader.channels:
        ax.plot(t_us, raw[ch][i] * 1e3, lw=0.8, label=ch)
    ax.set_title(f"event {i}", fontsize=9)
axes[0, 0].legend(fontsize=8)
fig.supxlabel("time [us]"); fig.supylabel("polarity-corrected [mV]"); fig.tight_layout()

In [ ]:
mean_wave = {ch: raw[ch][order[:50]].mean(0) for ch in reader.channels}
plt.figure()
for ch, m in mean_wave.items():
    plt.plot(t_us, m * 1e3, label=f"{ch}: 10-90% rise {rise_time(m, dt, ti) * 1e6:.1f} us")
plt.xlim(-10, 80); plt.xlabel("time [us]"); plt.ylabel("mV"); plt.legend()
plt.title("mean of the 50 largest events")

## 3. 減衰時定数 τ（pole-zero）

`tune_tau` は、フラットトップのうち電荷収集が終わった後ろ側だけで droop が 0 になる τ を探します。
旧 notebook の評価窓（フラットトップ先頭 +10%）も `skip=` で再現して並べます。
実際の波形は単一指数ではない（テールのフィットと大きく違う）ので、最終的には §5 の分解能で決めます。

In [ ]:
sh = processing["shaper"]
for ch in reader.channels:
    shaper = TrapezoidShaper.from_time(dt, sh["rise_us"] * 1e-6, sh["flat_us"] * 1e-6, 1.0)
    taus = np.geomspace(5e-6, 1000e-6, 200) / dt
    best, droops = tune_tau(raw[ch], shaper, taus, ti)
    old, _ = tune_tau(raw[ch], shaper, taus, ti, skip=max(5, int(0.1 * shaper.flat)))
    tail = fit_tail_tau(mean_wave[ch], reader.time, ti)
    print(f"{ch}: tau (late window) {best * dt * 1e6:6.1f} us | notebook window {old * dt * 1e6:6.1f} us | "
          f"tail fit {tail * 1e6:6.1f} us | used {processing['channels'][ch]['tau_us']} us")

## 4. 台形整形と波高の取り出し窓

フラットトップが本当に平らになるのは電荷収集が終わってからです。
TlBr では anode で 6 µs 程度、cathode で 10 µs 以上かかるため、12 µs のフラットトップ全体（15–85%）を平均すると、立ち上がりの途中を拾ってしまいます。
`flattop_window` を指定すると、後半だけを平均します（灰色の帯）。

In [ ]:
i = int(order[0])
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
for ch in reader.channels:
    cp = proc[ch]
    s = cp.shaper(raw[ch][i:i + 1])[0]
    axes[0].plot(t_us, raw[ch][i] * 1e3, lw=0.8, label=ch)
    axes[1].plot(t_us, s * 1e3, lw=0.9, label=f"{ch}: amp {cp.amplitude(reader.waveforms(ch, slice(i, i + 1)))[0] * 1e3:.1f} mV")
cp = proc[ANODE]
lo, hi = cp.shaper.flattop(ti)
w0, w1 = cp.window or (cp.guard_frac, 1 - cp.guard_frac)
axes[1].axvspan(t_us[lo], t_us[hi], color="k", alpha=0.05)
axes[1].axvspan(t_us[lo + int(w0 * cp.shaper.flat)], t_us[lo + int(w1 * cp.shaper.flat)], color="k", alpha=0.15)
axes[0].set_ylabel("raw [mV]"); axes[1].set_ylabel("shaped [mV]"); axes[1].set_xlabel("time [us]")
axes[0].legend(); axes[1].legend(); axes[1].set_xlim(-20, 100); fig.tight_layout()

## 5. 整形パラメータのスキャン

ノイズや droop の代わりに、深さ補正後の光電ピークの FWHM で比べます（測りたいのは結局これなので）。
1組あたり全イベントの整形を1回するので、組み合わせは少なめにします。`SCAN_EVENTS` で使うイベント数を絞れます。

In [ ]:
RUN_SCAN = True
SCAN_EVENTS = 8000
LINE_KEV = max(LINES[SRC0], key=lambda l: l[1])[0]       # 一番強いライン（Ba-133 なら 356 keV）
scan = [  # (flat_us, window, tau_A, tau_B)
    (12.0, None, 55.0, 17.0),        # 旧 notebook 相当
    (16.0, (0.6, 0.9), 55.0, 17.0),
    (20.0, (0.6, 0.9), 55.0, 17.0),
    (16.0, (0.6, 0.9), 45.0, 25.0),
]

def resolution(flat, window, tau_a, tau_b):
    p = deep_merge(processing, {"shaper": {"flat_us": flat}, "flattop_window": window,
                                "channels": {ANODE: {"tau_us": tau_a}, CATHODE: {"tau_us": tau_b}}})
    ef = EventFile.from_waveforms(files[SRC0][:1], processing=p, cuts=CUTS, max_events=SCAN_EVENTS, verbose=False)
    dc = DepthCorrection.fit(ef.amp("anode"), ef.column("CAR"), DEPTH["amp_window"], DEPTH["car_range"],
                             DEPTH["car_bins"], DEPTH["car_ref"], DEPTH["degree"])
    a = dc.apply(ef.amp("anode"), ef.column("CAR"))
    a = a[np.isfinite(a)]
    m = dc.amp_ref
    c, y, _ = histogram(a, bins=60, range=(0.94 * m, 1.05 * m))
    f = fit_peak(c, y, 0.94 * m, 1.05 * m)
    return 100 * f.fwhm / f.mu, np.isfinite(dc.apply(ef.amp("anode"), ef.column("CAR"))).mean()

if RUN_SCAN:
    for row in scan:
        try:
            fw, frac = resolution(*row)
            print(f"flat {row[0]:4.0f} us, window {row[1]}, tau {row[2]:g}/{row[3]:g}: "
                  f"{LINE_KEV:g} keV FWHM {fw:.2f} %, in CAR range {100 * frac:.0f} %")
        except RuntimeError as e:
            print(row, "failed:", e)

## 6. 全イベントの処理 → イベントファイル

In [ ]:
efs = {}
for src, fs in files.items():
    fn = OUT / f"{src}_events.fits"
    efs[src] = EventFile.from_waveforms(fs, processing=processing, cuts=CUTS, max_events=MAX_EVENTS)
    print(src, len(efs[src]), "events")
ef = efs[SRC0]
ef.events[:5]

In [ ]:
# 品質カットの変数（しきい値は CUTS、変えたら ef.recompute_good(**CUTS)）
ev = ef.events
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
axes[0].hist(np.asarray(ev[f"BLRMS_{ANODE}"]) * 1e3, bins=60); axes[0].set_xlabel("baseline RMS [mV]")
axes[1].hist(np.asarray(ev[f"BLSLOPE_{ANODE}"]) * ef.meta["samples"] * 1e3, bins=60); axes[1].set_xlabel("baseline drift [mV]")
axes[2].hist(np.asarray(ev[f"LATE_{ANODE}"]) * 1e3, bins=60); axes[2].set_xlabel("late-pulse indicator [mV]")
for ax in axes: ax.set_yscale("log")
fig.tight_layout()
for src, e in efs.items():
    print(f"{src}: good {np.mean(e.events['GOOD']) * 100:.1f} %")

## 7. スペクトル

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, ch in zip(axes, (ANODE, CATHODE)):
    c, y, _ = histogram(ef.amp(ch), bins=300)
    ax.step(c * 1e3, y, where="mid", lw=0.9)
    ax.set_xlabel(f"{ch} pulse height [mV]"); ax.set_ylabel("counts / bin")
fig.suptitle(f"{SRC0}: before depth correction"); fig.tight_layout()

## 8. CAR による深さ補正

一番強いラインを CAR のスライスごとにフィットし、多項式で補正曲線を作ります。
`amp_window` は広めで大丈夫です。各スライスでは窓の中の最大ピークの周りだけをフィットし、曲線から外れたスライスは自動的に外します（白丸が残ったスライス）。

In [ ]:
dc = DepthCorrection.fit(ef.amp("anode"), ef.column("CAR"), DEPTH["amp_window"], DEPTH["car_range"],
                         DEPTH["car_bins"], DEPTH["car_ref"], DEPTH["degree"])
print(f"CAR {dc.car_lim[0]:.2f}-{dc.car_lim[1]:.2f}, peak at CAR_REF {dc.amp_ref * 1e3:.2f} mV")
print("peak per slice [mV]:", np.round(dc.peak * 1e3, 1))
depth_figure(ef, dc);

## 9. エネルギー校正

`calibrate` は、まず Cs-137 か Ba-133 の一番強いピークから大まかなゲインを決めます（`approx` を与えればそれを使います）。
その後、全線源のラインを2回測り直してフィットし、残差が大きい点（別のピークを拾ったもの）を外します。
モデルは `linear0`（原点を通る直線）、`linear`、`quadratic` から選べます。

In [ ]:
spectra = {}
for src, e in efs.items():
    e.apply_depth(dc)
    spectra[src] = e.selected("AMP_CORR")
cal = calibrate(spectra, CAL["model"], CAL["approx"], min_amp=CAL["min_amp"])
print(cal.summary())
if cal.note:
    print("(" + cal.note + ")")
calibration_figure(cal);

## 10. 校正済みスペクトルと分解能

In [ ]:
for e in efs.values():
    e.apply_calibration(cal)
spectra_figure(efs);
for p, fw in zip(cal.points, cal.fwhm_keV()):
    print(f"{p.source:6s} {p.energy:8.2f} keV: FWHM {fw:5.1f} keV ({100 * fw / p.energy:.2f} %)")

## 11. 保存

In [ ]:
for src, e in efs.items():
    e.write(OUT / f"{src}_events.fits")     # EVENTS + FILES + DEPTH + CALIB
dc.write(OUT / "depth.fits")
cal.write(OUT / "calib.fits")
print(sorted(p.name for p in OUT.iterdir()))
# 読み直し: EventFile.read(OUT / "Ba133_events.fits"), EnergyCalibration.read(OUT / "calib.fits")

## メモ

- **波高の窓**：電荷収集（anode 約 6 µs、cathode 10 µs 以上）が終わってから平均すること。フラットトップを 16–20 µs にして、後半（0.6–0.9）を使う。
- **τ** は単一指数では決まらないので、§5 の分解能で選ぶ。
- **深さ補正**は CAR の範囲外（`DEPTH_OK = False`）のイベントを NaN にする。CAR が小さい側は曲線が急で、3次の多項式では追い切れないことがある。
- **時間変化**（分極によるゲインの変化）：新しいファイルには `TIME`・`EPOCH` 列がある。光電ピークの位置を時間のビンごとに追えば見られる。